[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/01_gpu_fundamentals/01.5_decode_and_batching/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue?logo=)](https://molab.cloud/github/harshuljain13/llm-inference-at-scale/blob/master/content/01_gpu_fundamentals/01.5_decode_and_batching/lab.ipynb)

# Lab 1.5: Decode and Batching

## Setup

This lab explores how batching transforms decode throughput from memory-bound to compute-bound.
We model a 7B parameter model (FP16) on an A100 80GB GPU.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# A100 80GB specs
BANDWIDTH_TB_S = 2.0  # TB/s HBM bandwidth
BANDWIDTH_B_S = BANDWIDTH_TB_S * 1e12  # bytes/s
FLOPS = 312e12  # FP16 TFLOPS -> FLOPS

# 7B model in FP16
MODEL_PARAMS = 7e9
BYTES_PER_PARAM = 2  # FP16
MODEL_SIZE_BYTES = MODEL_PARAMS * BYTES_PER_PARAM  # 14 GB

# KV cache params (Llama-2 7B style)
N_LAYERS = 32
N_KV_HEADS = 32  # MHA
D_HEAD = 128
SEQ_LEN = 2048  # context length

print(f"Model size: {MODEL_SIZE_BYTES/1e9:.1f} GB")
print(f"HBM bandwidth: {BANDWIDTH_TB_S} TB/s")
print(f"Peak FP16 FLOPS: {FLOPS/1e12:.0f} TFLOPS")

## Exercise 1: Theoretical Max Tokens/sec at Batch=1

At batch=1, decode is entirely memory-bound. Each token requires reading all model weights once.
The ceiling is: `tokens/sec = bandwidth / model_size`.

In [ ]:
# At batch=1, we must read all weights for each token
tokens_per_sec_b1 = BANDWIDTH_B_S / MODEL_SIZE_BYTES

print(f"Theoretical max tokens/sec (batch=1): {tokens_per_sec_b1:.1f}")
print(f"That's {1000/tokens_per_sec_b1:.2f} ms per token")
print(f"\nThis is the memory-bandwidth ceiling. No amount of")
print(f"compute optimization helps -- we're waiting on HBM reads.")

## Exercise 2: Arithmetic Intensity vs Batch Size

Arithmetic intensity = FLOPs per byte transferred.  
For decode: `AI = 2 * batch_size * params / (params * bytes_per_param) = batch_size`  
The roofline crossover (compute-bound threshold) is at `AI = FLOPS / bandwidth`.

In [ ]:
batch_sizes = np.arange(1, 257)

# FLOPs per token: ~2 * params (matmul)
flops_per_token = 2 * MODEL_PARAMS
# Bytes loaded per batch: model weights (amortized across batch)
bytes_per_batch = MODEL_SIZE_BYTES  # weights loaded once per batch

# Arithmetic intensity = total_flops / bytes_loaded
arithmetic_intensity = (flops_per_token * batch_sizes) / bytes_per_batch

# Roofline crossover point
machine_balance = FLOPS / BANDWIDTH_B_S  # FLOP/byte

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(batch_sizes, arithmetic_intensity, 'b-', linewidth=2, label='Arithmetic Intensity')
ax.axhline(y=machine_balance, color='r', linestyle='--', linewidth=2,
           label=f'Compute-bound threshold ({machine_balance:.0f} FLOP/byte)')
crossover_batch = int(np.ceil(machine_balance))
ax.axvline(x=crossover_batch, color='g', linestyle=':', linewidth=1.5,
           label=f'Crossover batch size = {crossover_batch}')
ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('Arithmetic Intensity (FLOP/byte)', fontsize=12)
ax.set_title('Decode Arithmetic Intensity vs Batch Size (7B FP16, A100)', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(0, 256)
ax.grid(True, alpha=0.3)
ax.fill_between(batch_sizes, 0, machine_balance, alpha=0.05, color='red', label='_')
ax.text(5, machine_balance * 0.4, 'MEMORY\nBOUND', fontsize=14, color='red', alpha=0.5)
ax.text(200, machine_balance * 1.2, 'COMPUTE\nBOUND', fontsize=14, color='blue', alpha=0.5)
plt.tight_layout()
plt.show()

print(f"\nCrossover batch size: {crossover_batch}")
print(f"Below this -> memory-bound (adding batch costs nothing)")
print(f"Above this -> compute-bound (adding batch costs latency)")

## Exercise 3: Diminishing Returns -- KV Cache Growth

Each batch element adds KV cache memory. As batch grows, total memory read per step
includes KV cache, eroding the "free" throughput gain from batching.

In [ ]:
def kv_cache_bytes(batch_size, seq_len=SEQ_LEN):
    """KV cache size in bytes for the full batch."""
    # 2 (K+V) * layers * kv_heads * d_head * seq_len * batch * bytes
    return 2 * N_LAYERS * N_KV_HEADS * D_HEAD * seq_len * batch_size * BYTES_PER_PARAM

batches = np.arange(1, 513)
kv_sizes = np.array([kv_cache_bytes(b) for b in batches])
total_read = MODEL_SIZE_BYTES + kv_sizes  # weights + KV per step

# Effective tokens/sec accounting for KV cache reads
effective_tps = (BANDWIDTH_B_S / total_read) * batches
# Ideal (no KV overhead)
ideal_tps = (BANDWIDTH_B_S / MODEL_SIZE_BYTES) * batches

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(batches, ideal_tps, 'b--', linewidth=1.5, label='Ideal (no KV overhead)', alpha=0.7)
ax1.plot(batches, effective_tps, 'r-', linewidth=2, label='With KV cache reads')
ax1.set_xlabel('Batch Size', fontsize=12)
ax1.set_ylabel('Tokens/sec', fontsize=12)
ax1.set_title('Throughput: Ideal vs Reality', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Efficiency = actual / ideal
efficiency = effective_tps / ideal_tps * 100
ax2.plot(batches, efficiency, 'g-', linewidth=2)
ax2.axhline(y=50, color='r', linestyle='--', alpha=0.5, label='50% efficiency')
ax2.set_xlabel('Batch Size', fontsize=12)
ax2.set_ylabel('Batching Efficiency (%)', fontsize=12)
ax2.set_title('Diminishing Returns from KV Cache Growth', fontsize=13)
ax2.set_ylim(0, 105)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find where efficiency drops below 50%
half_eff_idx = np.argmax(efficiency < 50)
if half_eff_idx > 0:
    print(f"\nEfficiency drops below 50% at batch size {batches[half_eff_idx]}")
print(f"KV cache at batch=1: {kv_cache_bytes(1)/1e9:.2f} GB")
print(f"KV cache at batch=64: {kv_cache_bytes(64)/1e9:.2f} GB")
print(f"KV cache at batch=256: {kv_cache_bytes(256)/1e9:.2f} GB")

## Exercise 4: Finding the Batch Size Where KV Cache Exceeds Weights

There is a critical batch size where KV cache becomes larger than the model weights.
Beyond this point, KV cache dominates memory bandwidth, and batching hurts more than helps.

In [ ]:
# Find batch where KV cache > model weights
# kv_cache_bytes(B) = 2 * 32 * 32 * 128 * 2048 * B * 2
kv_per_request = 2 * N_LAYERS * N_KV_HEADS * D_HEAD * SEQ_LEN * BYTES_PER_PARAM
critical_batch = MODEL_SIZE_BYTES / kv_per_request

print(f"KV cache per request (seq_len={SEQ_LEN}): {kv_per_request/1e9:.2f} GB")
print(f"Model weights: {MODEL_SIZE_BYTES/1e9:.1f} GB")
print(f"Critical batch size (KV = weights): {critical_batch:.1f}")
print(f"\nAt batch > {int(np.ceil(critical_batch))}, you read more KV cache than weights per step.")

# Visualize the crossover
batches_small = np.arange(1, 65)
kv_gb = np.array([kv_cache_bytes(b)/1e9 for b in batches_small])

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(batches_small, MODEL_SIZE_BYTES/1e9, alpha=0.3, color='blue',
       label=f'Model weights ({MODEL_SIZE_BYTES/1e9:.0f} GB)', width=1.0)
ax.plot(batches_small, kv_gb, 'r-', linewidth=2.5, label='KV cache size')
ax.axvline(x=critical_batch, color='black', linestyle='--', linewidth=1.5,
           label=f'Crossover (batch={critical_batch:.0f})')
ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('Memory (GB)', fontsize=12)
ax.set_title('KV Cache vs Model Weights (7B FP16, seq=2048)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Practical VRAM budget
vram = 80e9  # 80GB A100
available_for_kv = vram - MODEL_SIZE_BYTES - 2e9  # 2GB overhead
max_batch_vram = available_for_kv / kv_per_request
print(f"\n--- Practical VRAM Budget ---")
print(f"Available for KV cache: {available_for_kv/1e9:.1f} GB")
print(f"Max batch that fits in VRAM: {int(max_batch_vram)}")

## Key Takeaways

1. **Batch=1 ceiling**: Decode at batch=1 is purely memory-bound. Max throughput = bandwidth / model_size.
2. **Free throughput**: Below the roofline crossover, increasing batch is nearly free (same weights loaded, more tokens produced).
3. **KV cache is the tax**: Each batch element adds KV cache that must also be read, creating diminishing returns.
4. **Critical crossover**: When KV cache exceeds model weights, you're bandwidth-bound on cache reads, not weights.
5. **VRAM is the hard wall**: Even before bandwidth saturation, you run out of physical memory for KV cache.